# Day 3: Story Weaver - Adding Sentiment Analysis

**Objective**: Enhance the Story Weaver chatbot by adding sentiment analysis to adapt the story tone based on the user’s mood (e.g., happy, sad) using TextBlob. Build on the Day 2 choice system.

**Steps**:
1. Install and import TextBlob.
2. Load the model and story state from Day 2.
3. Get user mood and choice input.
4. Generate a story with tone adjusted by sentiment.
5. Save and document the results.

**Note**: Install `textblob` with `pip install textblob`.

## Install and set up textblob

In [1]:
# Install TextBlob (run this cell once)
# Import libraries
from transformers import pipeline
from textblob import TextBlob
import os

# Define file paths
project_root = os.path.join("..")
cleaned_file = os.path.join(project_root, "data", "processed", "wizard_of_oz_cleaned.txt")

# Verify cleaned file
if not os.path.exists(cleaned_file):
    print(f"Error: Cleaned file not found at {cleaned_file}. Run Day 1 notebook first.")
else:
    print(f"Cleaned file found: {cleaned_file}")

# Load DistilGPT-2
try:
    generator = pipeline('text-generation', model='distilgpt2')
    print("DistilGPT-2 model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {str(e)}")

Cleaned file found: ../data/processed/wizard_of_oz_cleaned.txt


Device set to use mps:0


DistilGPT-2 model loaded successfully!


## Initialize story and choices

In [12]:
# Initialize or load story state (continuing from Day 2)
story_state = {
    "current_scene": "crossroads",
    "choices_made": [],
    "last_prompt": "",
    "mood": "neutral"  # New field for sentiment
}

# Define available choices
choices = {
    "crossroads": {
        1: "Follow the golden road to the old temple",
        2: "Follow the golden road to the river bed where children live"
    }
}

print("Starting scene:", story_state["current_scene"])
print("Available choices:", choices[story_state["current_scene"]])

Starting scene: crossroads
Available choices: {1: 'Follow the golden road to the old temple', 2: 'Follow the golden road to the river bed where children live'}


## Get user mood and form story

In [13]:
def get_user_mood():
    while True:
        mood = input("How do you feel? (e.g., happy, sad, excited, neutral): ").lower()
        if mood in ["happy", "sad", "excited", "neutral"]:
            return mood
        print("Please enter a valid mood (happy, sad, excited, neutral).")

def get_user_choice():
    print("\nChoose your path:")
    for key, value in choices[story_state["current_scene"]].items():
        print(f"{key}: {value}")
    while True:
        try:
            choice = int(input("Enter choice (1 or 2): "))
            if choice in choices[story_state["current_scene"]]:
                return choice
            else:
                print("Invalid choice. Please enter 1 or 2.")
        except ValueError:
            print("Please enter a number (1 or 2).")

# Get mood and choice
story_state["mood"] = get_user_mood()
selected_choice = get_user_choice()
story_state["choices_made"].append(choices[story_state["current_scene"]][selected_choice])
print("You feel:", story_state["mood"])
print("You chose:", story_state["choices_made"][-1])


Choose your path:
1: Follow the golden road to the old temple
2: Follow the golden road to the river bed where children live
You feel: happy
You chose: Follow the golden road to the old temple


## generate the story with sentimental

In [14]:
# Define tone adjustments based on mood
tone_adjustments = {
    "happy": "with a joyful and bright atmosphere",
    "sad": "with a somber and reflective mood",
    "excited": "with an adventurous and thrilling vibe",
    "neutral": "with a calm and steady pace"
}

# Create prompt with integrated sentiment and stronger context
if selected_choice == 1:
    base_prompt = "Dorothy chose to follow the golden road to the old temple in the land of Oz. The temple stood ancient and mysterious, its walls glowing with magic..."
elif selected_choice == 2:
    base_prompt = "Dorothy chose to follow the golden road to the river bed where children lived in the land of Oz. The river sparkled with magical light..."

# Integrate mood seamlessly into the narrative
mood_context = tone_adjustments[story_state["mood"]]
new_prompt = f"{base_prompt} As she entered, the scene unfolded {mood_context}, revealing..."

# Generate story with anti-repetition
try:
    story = generator(new_prompt, max_length=250, num_return_sequences=1, temperature=0.7, 
                     no_repeat_ngram_size=2, truncation=True, pad_token_id=generator.tokenizer.eos_token_id)
    generated_story = story[0]['generated_text']
    print("\nGenerated Story:")
    print(generated_story)
    story_state["last_prompt"] = new_prompt
except Exception as e:
    print(f"Error generating story: {str(e)}")

# Update scene
story_state["current_scene"] = "temple" if selected_choice == 1 else "river_bed"
print("New scene:", story_state["current_scene"])


Generated Story:
Dorothy chose to follow the golden road to the old temple in the land of Oz. The temple stood ancient and mysterious, its walls glowing with magic... As she entered, the scene unfolded with a joyful and bright atmosphere, revealing...

Afterward, she smiled at the girl who was standing next to her, and took the opportunity to take her into the temple. Afterward when she saw the young man who had passed away, her eyes widened, as she asked who she was.
During her journey, however, this young girl, who happened to be known as the "Xian" or "Yuan" was unable to explain the matter. Her mind began to wander over this mysterious creature and all the other things. In contrast, when the light suddenly became dim and the world became dark, it was no longer able to understand what was happening. When the lights completely stopped, everything around her instantly disappeared and she looked at her face. She was not alone. This dark world would become a part of the human world. Th

## Saving the story

In [15]:
# Save story to output
output_file = os.path.join(project_root, "output", f"generated_story_{story_state['current_scene']}_{story_state['mood']}_2025-04-15.txt")
os.makedirs(os.path.join(project_root, "output"), exist_ok=True)
with open(output_file, 'w', encoding='utf-8') as file:
    file.write(generated_story)
print(f"Story saved to: {output_file}")

# Print state for debugging
print("Current story state:", story_state)

Story saved to: ../output/generated_story_temple_happy_2025-04-15.txt
Current story state: {'current_scene': 'temple', 'choices_made': ['Follow the golden road to the old temple'], 'last_prompt': 'Dorothy chose to follow the golden road to the old temple in the land of Oz. The temple stood ancient and mysterious, its walls glowing with magic... As she entered, the scene unfolded with a joyful and bright atmosphere, revealing...', 'mood': 'happy'}


## Next Steps for Day 3

**Done**:
- Added sentiment analysis with TextBlob to adjust story tone.
- Integrated mood input with the choice system.
- Generated and saved a sentiment-based story.

**To Do**:
- Test different moods and refine tone adjustments.
- Expand `choices` with sub-scenes (e.g., temple riddle).
- Prepare for Day 4: Fine-tune DistilGPT-2 with cleaned text.

**Action**: Commit changes to GitHub.